# BERT(MLM) wikitext 사전학습 (Google Colab)

번역용 Transformer의 인코더 부품을 재사용한 **encoder-only BERT**를 wikitext-2에서
Masked Language Modeling으로 사전학습한다.

**런타임 설정**: 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

## 1. GPU 확인

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 2. 저장소 클론 및 의존성 설치

In [ ]:
import os

if not os.path.exists('/content/MyTransformer'):
    !git clone https://github.com/youngpark-POS/MyTransformer.git
else:
    print('이미 클론됨, 최신 코드로 업데이트...')
    !git -C /content/MyTransformer pull

%cd /content/MyTransformer

In [ ]:
import os
# subprocess(!python ...)가 src 레이아웃 패키지를 찾을 수 있도록 PYTHONPATH 설정
# NOTE: pip install -e . 의 editable finder는 Colab에서 transformer 서브패키지(data/model/bert)를
#       노출하지 못해 ModuleNotFoundError를 일으키므로 사용하지 않고 PYTHONPATH만 쓴다.
os.environ['PYTHONPATH'] = '/content/MyTransformer/src'

# 혹시 이전 실행에서 남은 editable install이 PYTHONPATH를 가로채지 않도록 제거(있을 때만)
!pip uninstall -y transformer -q 2>/dev/null

# torch는 Colab에 기설치 — datasets, tqdm만 추가 설치 (matplotlib는 Colab 기본 제공)
!pip install -q datasets tqdm

## 3. MLM 사전학습

첫 실행 시 HuggingFace에서 **wikitext-2-raw-v1**를 다운로드하고 vocab 캐시
(`data/vocab_mlm.json`)를 생성한다. epoch마다 `checkpoints/bert_history.json`에
메트릭을 기록하고 best(val_loss 최소) 체크포인트를 `checkpoints/bert_best.pt`에 저장한다.

> `train_acc`/`val_acc`는 **마스킹된 위치**만 대상으로 한 토큰 예측 정확도다.

In [ ]:
!python -m transformer.bert.train --epochs 10

## 3-1. 학습 곡선 시각화 (loss / accuracy)

`bert/train.py`가 epoch마다 `checkpoints/bert_history.json`에 기록한 메트릭을 읽어
번역에서 쓰던 `plot_history.py`를 그대로 재사용해 곡선을 그린다(키 구조가 동일하다).

In [ ]:
import sys
sys.path.insert(0, '/content/MyTransformer')  # plot_history.py(프로젝트 루트) import 경로
from pathlib import Path
from plot_history import load_history, plot_history

history = load_history(Path('checkpoints/bert_history.json'))
plot_history(history, show=True)  # show=True면 Colab에 그림이 바로 렌더링된다

## 4. 체크포인트를 Google Drive에 저장

> Colab 세션이 끊기면 `/content` 내용이 사라집니다. Drive에 백업해두세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil, os
dst = '/content/drive/MyDrive/MyTransformer'
os.makedirs(dst, exist_ok=True)
shutil.copy('checkpoints/bert_best.pt', dst + '/bert_best.pt')
print('저장 완료:', dst + '/bert_best.pt')

## 5. 마스크 채우기(fill-mask) 추론

BERT는 번역이 아니라 **문맥으로 가려진 단어를 예측**하는 모델이다. 체크포인트를 복원해
`[MASK]` 자리에 들어갈 토큰 후보를 확률순으로 본다.

> wikitext-2는 작은 코퍼스라 완벽한 예측을 기대하긴 어렵지만, 문맥상 그럴듯한 후보가
> 상위에 오르는지로 학습이 됐는지 감을 잡을 수 있다.

In [ ]:
import sys
sys.path.insert(0, '/content/MyTransformer/src')
from pathlib import Path
import torch

from config import CLS_IDX, SEP_IDX, MASK_IDX, UNK_IDX
from transformer.bert.model import BertForMaskedLM
from transformer.data.vocab import Vocab
from transformer.data.tokenizer import tokenize
from transformer.utils import load_checkpoint, resolve_device

device = resolve_device('cuda')
ckpt = load_checkpoint(Path('checkpoints/bert_best.pt'), map_location=device)
cfg = ckpt['cfg']
vocab = Vocab(ckpt['itos'])

model = BertForMaskedLM(
    len(vocab),
    d_model=cfg.model.d_model,
    n_heads=cfg.model.n_heads,
    n_layers=cfg.model.n_layers,
    d_ff=cfg.model.d_ff,
    dropout=cfg.model.dropout,
    max_len=cfg.model.max_len,
    tie_embeddings=cfg.model.tie_embeddings,
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'모델 로드 완료 / vocab={len(vocab)}')

In [ ]:
@torch.no_grad()
def fill_mask(text, topk=5):
    """text 안의 '[MASK]' 자리에 들어갈 토큰 후보 topk개를 (토큰, 확률)로 반환."""
    left, _, right = text.partition('[MASK]')
    tokens = tokenize(left) + ['[MASK]'] + tokenize(right)
    ids = [CLS_IDX]
    for t in tokens:
        ids.append(MASK_IDX if t == '[MASK]' else vocab.stoi.get(t, UNK_IDX))
    ids.append(SEP_IDX)
    pos = ids.index(MASK_IDX)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    probs = model(x)[0, pos].softmax(-1)
    top = probs.topk(topk)
    return [(vocab.itos[i], round(probs[i].item(), 4)) for i in top.indices.tolist()]

for sent in [
    'the capital of france is [MASK] .',
    'he was born in the [MASK] of 1990 .',
    'the film received [MASK] reviews from critics .',
]:
    print(sent)
    for tok, p in fill_mask(sent):
        print(f'    {p:.4f}  {tok}')
    print()

## 6. (선택) Drive에서 체크포인트 불러와 추론

세션이 재시작된 경우, Drive에서 체크포인트를 복원한 뒤 위 5번 셀들을 다시 실행하면 된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('checkpoints', exist_ok=True)
shutil.copy('/content/drive/MyDrive/MyTransformer/bert_best.pt', 'checkpoints/bert_best.pt')
print('체크포인트 복원 완료')